# 05. Age Reference Construction

Build per-site linear age references for every modality x cell-type x batch group.

**Outputs** (`results/age_references_v2/`):
- `arrays/` -- full genome-wide numpy arrays (N, keep, coef, intercept, pearson_r)
- `compact/` -- eligible-site-only arrays for fast inference
- `reference_index.csv` -- registry of every saved reference

**Inputs** (from prior notebooks):
- Bulk `.beta` files from all 9 batches (`results/<batch_token>/<pid>.beta`)
- Cell-sorted `.beta` files from `results/celltype_sorted_v2/all7_v2/<CellType>/`

---

In [ ]:
# ============================================================
# MODE: switch between TEST (fast 3-sample smoke-test) and PRODUCTION
# ============================================================
MODE = "TEST"   # set to "PRODUCTION" for the full cohort run
TEST_SAMPLE_LIMIT = 3

print(f'Running in {MODE} mode')
if MODE == 'TEST':
    print(f'  TEST_SAMPLE_LIMIT = {TEST_SAMPLE_LIMIT} samples per group')


In [ ]:
import os
import io
import gc
import json
import numpy as np
import pandas as pd
from tqdm import tqdm
from google.cloud import storage

# ============================================================
# 0) CONFIG
# ============================================================
AGE_STEP = 0.1
EPS = 1e-3
CAP = 20.0

MIN_SAMPLES_PER_SITE = 20
DYNAMIC_MIN_FRAC = 0.50
ABSR_MIN = 0.30
RANGE_MIN = 0.05
MIN_GROUP_N = 20

TRAIN_FRAC = 0.80
SPLIT_SEED = 20260414

QC_METRICS_PATH = "gs://fc-aou-datasets-controlled/v8/wgs/short_read/snpindel/aux/qc/genomic_metrics.tsv"

WORKSPACE_BUCKET = os.environ["WORKSPACE_BUCKET"]
WORKSPACE_CDR = os.environ["WORKSPACE_CDR"]
GOOGLE_PROJECT = os.environ["GOOGLE_PROJECT"]

assert WORKSPACE_BUCKET.startswith("gs://")
BUCKET_NAME = WORKSPACE_BUCKET.replace("gs://", "")

OUTPUT_ROOT = "results/age_references_v2"
REF_OUT_PREFIX = f"{WORKSPACE_BUCKET}/{OUTPUT_ROOT}"
REFERENCE_INDEX_GS = f"{REF_OUT_PREFIX}/reference_index.csv"

LOCAL_TMP_ROOT = "/home/jupyter/reference_tmp_age_refs_v2"

CELL_TYPES = [
    "Myeloid",
    "Lymphoid",
    "T_Cell",
    "B_Cell",
    "NK_Cell",
    "Monocyte",
    "Granulocyte",
]

REFERENCE_GROUPS = {
    "bulk": [
        f"{WORKSPACE_BUCKET}/results/bi_sequel",
        f"{WORKSPACE_BUCKET}/results/bi_revio",
        f"{WORKSPACE_BUCKET}/results/bcm_revio",
        f"{WORKSPACE_BUCKET}/results/bcm_sequel",
        f"{WORKSPACE_BUCKET}/results/bcm_ont",
        f"{WORKSPACE_BUCKET}/results/jhu_ont",
        f"{WORKSPACE_BUCKET}/results/uw_revio",
        f"{WORKSPACE_BUCKET}/results/uw_sequel",
        f"{WORKSPACE_BUCKET}/results/ha_revio",
        # v7 intentionally excluded
        # f"{WORKSPACE_BUCKET}/results/v7_base",
    ],
    "Myeloid": [
        f"{WORKSPACE_BUCKET}/results/celltype_sorted_v2/all7_v2/Myeloid",
    ],
    "Lymphoid": [
        f"{WORKSPACE_BUCKET}/results/celltype_sorted_v2/all7_v2/Lymphoid",
    ],
    "T_Cell": [
        f"{WORKSPACE_BUCKET}/results/celltype_sorted_v2/all7_v2/T_Cell",
    ],
    "B_Cell": [
        f"{WORKSPACE_BUCKET}/results/celltype_sorted_v2/all7_v2/B_Cell",
    ],
    "NK_Cell": [
        f"{WORKSPACE_BUCKET}/results/celltype_sorted_v2/all7_v2/NK_Cell",
    ],
    "Monocyte": [
        f"{WORKSPACE_BUCKET}/results/celltype_sorted_v2/all7_v2/Monocyte",
    ],
    "Granulocyte": [
        f"{WORKSPACE_BUCKET}/results/celltype_sorted_v2/all7_v2/Granulocyte",
    ],
}

BATCH_TOKEN_TO_SOURCE_PREFIX = {
    "bi_sequel": f"{WORKSPACE_BUCKET}/results/bi_sequel",
    "bi_revio": f"{WORKSPACE_BUCKET}/results/bi_revio",
    "bcm_revio": f"{WORKSPACE_BUCKET}/results/bcm_revio",
    "bcm_sequel": f"{WORKSPACE_BUCKET}/results/bcm_sequel",
    "bcm_ont": f"{WORKSPACE_BUCKET}/results/bcm_ont",
    "jhu_ont": f"{WORKSPACE_BUCKET}/results/jhu_ont",
    "uw_revio": f"{WORKSPACE_BUCKET}/results/uw_revio",
    "uw_sequel": f"{WORKSPACE_BUCKET}/results/uw_sequel",
    "ha_revio": f"{WORKSPACE_BUCKET}/results/ha_revio",
    "v7_base": f"{WORKSPACE_BUCKET}/results/v7_base",
}

client = storage.Client()

print("WORKSPACE_BUCKET:", WORKSPACE_BUCKET)
print("BUCKET_NAME:", BUCKET_NAME)
print("WORKSPACE_CDR:", WORKSPACE_CDR)
print("GOOGLE_PROJECT:", GOOGLE_PROJECT)
print("REF_OUT_PREFIX:", REF_OUT_PREFIX)

### GCS and utility helpers

In [ ]:
# ============================================================
# 1) HELPERS
# ============================================================
def clear_large_objects(*objs):
    for obj in objs:
        try:
            del obj
        except Exception:
            pass
    gc.collect()

def sanitize_name(x: str) -> str:
    out = str(x)
    out = out.replace("gs://", "")
    out = out.replace("/", "__")
    out = out.replace(" ", "_")
    out = out.replace(":", "_")
    return out

def assign_technology_strict(prefix: str) -> str:
    p = prefix.lower()
    if "v7" in p:
        return "v7_base"
    if "ont" in p:
        return "ONT"
    if "revio" in p:
        return "PacBio_Revio"
    if "sequel" in p:
        return "PacBio_Sequel"
    return "Other_Unknown"

def build_age_grid_from_train(train_df, age_step=0.1):
    age_lo = float(np.floor(train_df.Age.min() / age_step) * age_step)
    age_hi = float(np.ceil(train_df.Age.max() / age_step) * age_step)
    return np.arange(age_lo, age_hi + 1e-9, age_step).astype(np.float32)

def gs_to_bucket_blob(gs_path: str):
    assert gs_path.startswith("gs://"), gs_path
    rest = gs_path[5:]
    bkt, obj = rest.split("/", 1)
    return bkt, obj

def gcs_blob_exists(gs_path: str) -> bool:
    bkt, obj = gs_to_bucket_blob(gs_path)
    blob = client.bucket(bkt).blob(obj)
    return blob.exists()

def upload_text_to_gcs(gs_path: str, text: str, content_type="text/plain", timeout=300):
    bkt, obj = gs_to_bucket_blob(gs_path)
    blob = client.bucket(bkt).blob(obj)
    blob.upload_from_string(text.encode("utf-8"), content_type=content_type, timeout=timeout)

def upload_df_to_gcs_csv(df: pd.DataFrame, gs_path: str, timeout=300):
    upload_text_to_gcs(gs_path, df.to_csv(index=False), content_type="text/csv", timeout=timeout)

def download_text_from_gcs(gs_path: str):
    bkt, obj = gs_to_bucket_blob(gs_path)
    blob = client.bucket(bkt).blob(obj)
    if not blob.exists():
        return None
    return blob.download_as_text()

def load_json_from_gcs(gs_path: str):
    txt = download_text_from_gcs(gs_path)
    if txt is None:
        raise FileNotFoundError(gs_path)
    return json.loads(txt)

def read_csv_from_gcs_text(gs_path: str) -> pd.DataFrame:
    return pd.read_csv(io.StringIO(download_text_from_gcs(gs_path)))

def upload_file_to_gcs(local_path: str, gs_path: str, content_type="application/octet-stream", timeout=1800):
    bkt, obj = gs_to_bucket_blob(gs_path)
    blob = client.bucket(bkt).blob(obj)
    blob.chunk_size = 16 * 1024 * 1024
    with open(local_path, "rb") as f:
        blob.upload_from_file(f, content_type=content_type, timeout=timeout)

def save_array_local_then_upload(arr: np.ndarray, gs_path: str, local_dir: str, name: str, allow_skip=True):
    os.makedirs(local_dir, exist_ok=True)

    if allow_skip and gcs_blob_exists(gs_path):
        print(f"Skip existing: {gs_path}")
        return gs_path

    local_path = os.path.join(local_dir, f"{name}.npy")
    np.save(local_path, arr, allow_pickle=False)

    print(f"Uploading {name} -> {gs_path}")
    upload_file_to_gcs(local_path, gs_path, timeout=1800)

    try:
        os.remove(local_path)
    except Exception:
        pass

    gc.collect()
    return gs_path

def load_npy_from_gcs(gs_path: str):
    bkt, obj = gs_to_bucket_blob(gs_path)
    blob = client.bucket(bkt).blob(obj)
    data = blob.download_as_bytes()
    return np.load(io.BytesIO(data), allow_pickle=False)

def load_beta_pairs_from_gcs(gs_path: str, dtype=np.uint8, min_bytes=1000) -> np.ndarray:
    assert gs_path.startswith("gs://"), gs_path
    bkt, obj = gs_to_bucket_blob(gs_path)
    blob = client.bucket(bkt).blob(obj)
    data = blob.download_as_bytes()
    if data is None or len(data) < min_bytes:
        raise ValueError(f"Empty/tiny beta file: {gs_path} bytes={0 if data is None else len(data)}")
    arr = np.frombuffer(data, dtype=dtype)
    if arr.size % 2:
        arr = arr[:-1]
    pairs = arr.reshape(-1, 2)
    if pairs.shape[0] == 0:
        raise ValueError(f"Parsed 0 CpGs from beta: {gs_path} bytes={len(data)}")
    return pairs

def deterministic_train_test_split(
    manifest_df: pd.DataFrame,
    train_frac: float = 0.8,
    seed: int = 20260414
):
    manifest_df = manifest_df.copy()

    unique_people = (
        manifest_df[["person_id"]]
        .drop_duplicates()
        .sort_values("person_id")
        .reset_index(drop=True)
    )

    rng = np.random.RandomState(seed)
    perm = rng.permutation(len(unique_people))
    n_train = int(np.floor(train_frac * len(unique_people)))

    train_ids = set(unique_people.iloc[perm[:n_train]]["person_id"].astype(str))
    test_ids  = set(unique_people.iloc[perm[n_train:]]["person_id"].astype(str))

    train_df = manifest_df[manifest_df["person_id"].astype(str).isin(train_ids)].copy()
    test_df  = manifest_df[manifest_df["person_id"].astype(str).isin(test_ids)].copy()

    train_df = train_df.sort_values(["source_prefix", "person_id"]).reset_index(drop=True)
    test_df  = test_df.sort_values(["source_prefix", "person_id"]).reset_index(drop=True)

    overlap = set(train_df["person_id"]).intersection(set(test_df["person_id"]))
    if len(overlap) > 0:
        raise RuntimeError(f"Train/test overlap detected: {len(overlap)} people")

    return train_df, test_df

### Clinical data: BigQuery demographics + exact age at blood draw

In [ ]:
# ============================================================
# 2) BUILD clinical_df WITH EXACT AGE AT DRAW
# ============================================================
dataset_36069554_person_sql = """
    SELECT
        person.person_id,
        person.gender_concept_id,
        p_gender_concept.concept_name as gender,
        person.birth_datetime as date_of_birth,
        person.race_concept_id,
        p_race_concept.concept_name as race,
        person.ethnicity_concept_id,
        p_ethnicity_concept.concept_name as ethnicity,
        person.sex_at_birth_concept_id,
        p_sex_at_birth_concept.concept_name as sex_at_birth,
        person.self_reported_category_concept_id,
        p_self_reported_category_concept.concept_name as self_reported_category
    FROM
        `""" + os.environ["WORKSPACE_CDR"] + """.person` person
    LEFT JOIN `""" + os.environ["WORKSPACE_CDR"] + """.concept` p_gender_concept
        ON person.gender_concept_id = p_gender_concept.concept_id
    LEFT JOIN `""" + os.environ["WORKSPACE_CDR"] + """.concept` p_race_concept
        ON person.race_concept_id = p_race_concept.concept_id
    LEFT JOIN `""" + os.environ["WORKSPACE_CDR"] + """.concept` p_ethnicity_concept
        ON person.ethnicity_concept_id = p_ethnicity_concept.concept_id
    LEFT JOIN `""" + os.environ["WORKSPACE_CDR"] + """.concept` p_sex_at_birth_concept
        ON person.sex_at_birth_concept_id = p_sex_at_birth_concept.concept_id
    LEFT JOIN `""" + os.environ["WORKSPACE_CDR"] + """.concept` p_self_reported_category_concept
        ON person.self_reported_category_concept_id = p_self_reported_category_concept.concept_id
    WHERE
        person.PERSON_ID IN (
            SELECT distinct person_id
            FROM `""" + os.environ["WORKSPACE_CDR"] + """.cb_search_person`
            WHERE has_lr_whole_genome_variant = 1
        )
"""

print("Querying demographics from BigQuery...")
dataset_36069554_person_df = pd.read_gbq(
    dataset_36069554_person_sql,
    dialect="standard",
    use_bqstorage_api=("BIGQUERY_STORAGE_API_ENABLED" in os.environ),
    progress_bar_type="tqdm_notebook"
)

print(f"Loading QC metrics from: {QC_METRICS_PATH}")
qc_df = pd.read_csv(
    QC_METRICS_PATH,
    sep="\t",
    storage_options={
        "requester_pays": True,
        "project": os.environ["GOOGLE_PROJECT"]
    }
)

if "sample_name" in qc_df.columns:
    qc_df = qc_df.rename(columns={"sample_name": "person_id"})
elif "research_id" in qc_df.columns:
    qc_df = qc_df.rename(columns={"research_id": "person_id"})
else:
    raise ValueError("Could not find sample_name or research_id in qc_df")

dataset_36069554_person_df["person_id"] = dataset_36069554_person_df["person_id"].astype(str)
qc_df["person_id"] = qc_df["person_id"].astype(str)

dataset_36069554_person_df = pd.merge(
    dataset_36069554_person_df,
    qc_df[["person_id", "biosample_collection_date"]],
    on="person_id",
    how="inner"
)

print("Calculating exact age at blood draw...")
dataset_36069554_person_df["date_of_birth"] = pd.to_datetime(
    dataset_36069554_person_df["date_of_birth"], utc=True
)
dataset_36069554_person_df["biosample_collection_date"] = pd.to_datetime(
    dataset_36069554_person_df["biosample_collection_date"], utc=True
)
dataset_36069554_person_df["age"] = (
    dataset_36069554_person_df["biosample_collection_date"]
    - dataset_36069554_person_df["date_of_birth"]
).dt.days / 365.25

clinical_df = dataset_36069554_person_df.copy()
clinical_df["person_id"] = clinical_df["person_id"].astype(str)
clinical_df["Age"] = pd.to_numeric(clinical_df["age"], errors="coerce")
clinical_df = clinical_df.dropna(subset=["Age"])
clinical_df = clinical_df[clinical_df["Age"] > 0].copy()

clinical_df = clinical_df[[
    "person_id",
    "Age",
    "sex_at_birth",
    "biosample_collection_date"
]].drop_duplicates(subset=["person_id"]).reset_index(drop=True)

print("Clinical participants with usable age:", len(clinical_df))

### Manifest discovery: bulk and cell-type beta files

In [ ]:
# ============================================================
# 3) DISCOVER MANIFESTS
# ============================================================
def discover_bulk_manifest():
    rows = []
    for bulk_prefix_gs in REFERENCE_GROUPS["bulk"]:
        assert bulk_prefix_gs.startswith(f"gs://{BUCKET_NAME}/"), bulk_prefix_gs
        prefix_no_gs = bulk_prefix_gs.replace(f"gs://{BUCKET_NAME}/", "")

        print(f"Scanning: {bulk_prefix_gs}")
        for blob in client.list_blobs(BUCKET_NAME, prefix=prefix_no_gs):
            if not blob.name.endswith(".beta"):
                continue

            pid = blob.name.split("/")[-1].replace(".beta", "")
            rows.append({
                "person_id": str(pid),
                "bulk_beta_path": f"gs://{BUCKET_NAME}/{blob.name}",
                "source_prefix": bulk_prefix_gs,
                "size_bytes": int(blob.size or 0)
            })

    df = pd.DataFrame(rows)
    if len(df) == 0:
        raise ValueError("No bulk beta files found")

    df = df[df["size_bytes"] >= 1000].copy()
    df = df.drop_duplicates(subset=["source_prefix", "person_id"])
    df = df.merge(clinical_df[["person_id", "Age"]], on="person_id", how="inner")
    df = df.sort_values(["source_prefix", "person_id"]).reset_index(drop=True)

    df["batch_group"] = df["source_prefix"].astype(str)
    df["technology_group"] = df["source_prefix"].apply(assign_technology_strict)

    print("Bulk beta samples after merge with clinical age:", len(df))
    print("\nCounts by source_prefix:")
    display(df.groupby("source_prefix")["person_id"].nunique().rename("n_files").to_frame())
    return df

def parse_celltype_filename(fname: str, celltype: str):
    """
    Expected filename pattern:
      <batch_token>__<person_id>_<celltype>.beta
    Example:
      bcm_revio__1005412_Myeloid.beta
    """
    suffix = f"_{celltype}.beta"
    if not fname.endswith(suffix):
        return None

    core = fname[:-len(suffix)]
    if "__" not in core:
        return None

    batch_token, person_id = core.split("__", 1)
    if len(person_id) == 0:
        return None

    return {
        "batch_token": batch_token,
        "person_id": str(person_id),
    }

def discover_celltype_files(celltype: str) -> pd.DataFrame:
    if celltype not in REFERENCE_GROUPS:
        raise ValueError(f"{celltype} not found in REFERENCE_GROUPS")

    rows = []
    prefixes = REFERENCE_GROUPS[celltype]

    print(f"\nScanning actual bucket files for {celltype}...")
    for cell_prefix_gs in prefixes:
        assert cell_prefix_gs.startswith(f"gs://{BUCKET_NAME}/"), cell_prefix_gs
        prefix_no_gs = cell_prefix_gs.replace(f"gs://{BUCKET_NAME}/", "")

        print("  prefix:", cell_prefix_gs)

        for blob in client.list_blobs(BUCKET_NAME, prefix=prefix_no_gs):
            if not blob.name.endswith(".beta"):
                continue

            fname = blob.name.split("/")[-1]
            parsed = parse_celltype_filename(fname, celltype)
            if parsed is None:
                continue

            rows.append({
                "raw_file_id": fname.replace(f"_{celltype}.beta", ""),
                "person_id": parsed["person_id"],
                "batch_token": parsed["batch_token"],
                "source_prefix_from_file": BATCH_TOKEN_TO_SOURCE_PREFIX.get(parsed["batch_token"], None),
                "celltype": celltype,
                "celltype_beta_path": f"gs://{BUCKET_NAME}/{blob.name}",
                "celltype_blob_name": blob.name,
                "size_bytes": int(blob.size or 0),
            })

    df = pd.DataFrame(rows)
    if len(df) == 0:
        print(f"No actual files found for {celltype}")
        return df

    df = df[df["size_bytes"] >= 1000].copy()
    df = df.drop_duplicates(subset=["raw_file_id", "celltype_beta_path"]).reset_index(drop=True)

    print(f"Found {len(df)} raw beta files for {celltype}")
    display(df.head(5))
    return df

def build_celltype_manifest_from_bulk(bulk_manifest: pd.DataFrame, celltype: str) -> pd.DataFrame:
    ct_files = discover_celltype_files(celltype)

    if len(ct_files) == 0:
        return ct_files

    # skip v7 immediately
    ct_files = ct_files[ct_files["batch_token"] != "v7_base"].copy()

    df = bulk_manifest.merge(
        ct_files[
            ["person_id", "batch_token", "source_prefix_from_file", "celltype", "celltype_beta_path"]
        ],
        on="person_id",
        how="inner"
    ).copy()

    # require batch encoded in filename to agree with bulk source_prefix
    df = df[df["source_prefix"] == df["source_prefix_from_file"]].copy()

    df = df.sort_values(["source_prefix", "person_id", "celltype_beta_path"]).reset_index(drop=True)
    df = df.drop_duplicates(subset=["source_prefix", "person_id"]).reset_index(drop=True)

    print(f"\nCell type {celltype}: matched {len(df)} files after join to bulk manifest")

    if len(df) > 0:
        print("Counts by source_prefix:")
        display(df.groupby("source_prefix")["person_id"].nunique().rename("n_files").to_frame())

        print("Counts by technology:")
        display(df.groupby("technology_group")["person_id"].nunique().rename("n_files").to_frame())

    return df

### Reference construction: per-site linear regression on training betas

In [ ]:
# ============================================================
# 4) REFERENCE CONSTRUCTION
# ============================================================
def construct_reference_from_betas(manifest_df: pd.DataFrame, beta_col: str, min_samples_per_site: int = 20):
    N = Sx = Sy = Sxx = Syy = Sxy = None
    n_sites = None

    for row in tqdm(manifest_df.itertuples(index=False), total=len(manifest_df), desc=f"Ref {beta_col}"):
        age = float(row.Age)
        pairs = load_beta_pairs_from_gcs(getattr(row, beta_col), dtype=np.uint8)

        if n_sites is None:
            n_sites = pairs.shape[0]
            print("Initialized arrays n_sites:", n_sites)
            N   = np.zeros(n_sites, dtype=np.float32)
            Sx  = np.zeros(n_sites, dtype=np.float32)
            Sy  = np.zeros(n_sites, dtype=np.float32)
            Sxx = np.zeros(n_sites, dtype=np.float32)
            Syy = np.zeros(n_sites, dtype=np.float32)
            Sxy = np.zeros(n_sites, dtype=np.float32)
        else:
            if pairs.shape[0] != n_sites:
                raise ValueError(
                    f"Inconsistent CpG length. Expected {n_sites}, got {pairs.shape[0]} "
                    f"for {getattr(row, beta_col)}"
                )

        m = pairs[:, 0].astype(np.float32)
        t = pairs[:, 1].astype(np.float32)
        valid = t > 0

        beta = np.zeros_like(m)
        np.divide(m, t, out=beta, where=valid)

        N[valid]   += 1.0
        Sx[valid]  += age
        Sy[valid]  += beta[valid]
        Sxx[valid] += age * age
        Syy[valid] += beta[valid] * beta[valid]
        Sxy[valid] += age * beta[valid]

        del pairs, m, t, valid, beta

    keep = N >= float(min_samples_per_site)

    denom_x = (N * Sxx) - (Sx * Sx)
    numer   = (N * Sxy) - (Sx * Sy)

    coef = np.zeros(n_sites, dtype=np.float32)
    intercept = np.zeros(n_sites, dtype=np.float32)
    R = np.zeros(n_sites, dtype=np.float32)
    t_stat = np.zeros(n_sites, dtype=np.float32)

    coef[keep] = np.divide(
        numer[keep],
        denom_x[keep],
        out=np.zeros_like(numer[keep]),
        where=denom_x[keep] != 0
    )
    intercept[keep] = np.divide(
        Sy[keep] - coef[keep] * Sx[keep],
        N[keep],
        out=np.zeros_like(Sy[keep]),
        where=N[keep] != 0
    )

    denom_y = (N * Syy) - (Sy * Sy)
    r_denom = np.sqrt(np.maximum(denom_x * denom_y, 0))
    R[keep] = np.divide(
        numer[keep],
        r_denom[keep],
        out=np.zeros_like(numer[keep]),
        where=r_denom[keep] != 0
    )

    r_sq = np.clip(R[keep] * R[keep], 0.0, 0.999999)
    dfree = np.maximum(N[keep] - 2.0, 1.0)
    t_stat[keep] = np.abs(R[keep]) * np.sqrt(dfree / (1.0 - r_sq))

    return {
        "mode": "dense",
        "N": N,
        "keep": keep,
        "pearson_r": R,
        "absR": np.abs(R),
        "t_stat": t_stat,
        "intercept": intercept,
        "coef": coef,
        "EPS": EPS,
        "CAP": CAP,
    }

def build_eligible_mask_endpoint_safe(ref, age_lo, age_hi, absR_min=0.30, range_min=0.05):
    ok = ref["keep"].copy()
    ok &= np.isfinite(ref["absR"])
    ok &= np.isfinite(ref["t_stat"])
    ok &= np.isfinite(ref["intercept"])
    ok &= np.isfinite(ref["coef"])

    if absR_min is not None:
        ok &= (ref["absR"] >= absR_min)

    p_lo = ref["intercept"] + ref["coef"] * float(age_lo)
    p_hi = ref["intercept"] + ref["coef"] * float(age_hi)

    ok &= (p_lo > ref["EPS"]) & (p_lo < 1.0 - ref["EPS"])
    ok &= (p_hi > ref["EPS"]) & (p_hi < 1.0 - ref["EPS"])

    if range_min is not None:
        expected_change = np.abs(p_hi - p_lo)
        ok &= (expected_change >= range_min)

    return ok

### Reference index helpers: load / upsert / check existence

In [ ]:
# ============================================================
# 5) INDEX HELPERS
# ============================================================
INDEX_COLUMNS = [
    "modality",
    "celltype",
    "group_level",
    "group_name",
    "split_seed",
    "train_frac",
    "n_total_samples",
    "n_train_samples",
    "n_test_samples",
    "age_min_train",
    "age_max_train",
    "age_grid_min",
    "age_grid_max",
    "age_step",
    "dynamic_min_samples",
    "n_sites_total",
    "n_sites_keep",
    "n_sites_eligible",
    "arrays_prefix",
    "compact_prefix",
    "meta_gs",
    "train_manifest_gs",
    "test_manifest_gs",
]

def load_reference_index():
    txt = download_text_from_gcs(REFERENCE_INDEX_GS)
    if txt is None or txt.strip() == "":
        return pd.DataFrame(columns=INDEX_COLUMNS)
    return pd.read_csv(io.StringIO(txt))

def upsert_reference_index_row(row_dict: dict):
    idx = load_reference_index()

    row_df = pd.DataFrame([row_dict])
    for c in INDEX_COLUMNS:
        if c not in row_df.columns:
            row_df[c] = np.nan
    row_df = row_df[INDEX_COLUMNS]

    if len(idx) > 0:
        keep_mask = ~(
            (idx["modality"].astype(str) == str(row_dict["modality"])) &
            (idx["celltype"].astype(str) == str(row_dict["celltype"])) &
            (idx["group_level"].astype(str) == str(row_dict["group_level"])) &
            (idx["group_name"].astype(str) == str(row_dict["group_name"]))
        )
        idx = idx.loc[keep_mask].copy()

    idx = pd.concat([idx, row_df], ignore_index=True)
    idx = idx.sort_values(["modality", "celltype", "group_level", "group_name"]).reset_index(drop=True)
    upload_df_to_gcs_csv(idx, REFERENCE_INDEX_GS)

def reference_already_exists(modality: str, celltype: str, group_level: str, group_name: str) -> bool:
    idx = load_reference_index()
    if len(idx) == 0:
        return False
    hit = idx[
        (idx["modality"].astype(str) == str(modality)) &
        (idx["celltype"].astype(str) == str(celltype)) &
        (idx["group_level"].astype(str) == str(group_level)) &
        (idx["group_name"].astype(str) == str(group_name))
    ]
    return len(hit) > 0

### Save reference bundle: upload arrays + compact representation to GCS

In [ ]:
# ============================================================
# 6) SAVE REFERENCE, CHUNKED
# ============================================================
def save_reference_bundle(
    ref: dict,
    train_df: pd.DataFrame,
    test_df: pd.DataFrame,
    age_grid: np.ndarray,
    modality: str,
    celltype: str,
    group_level: str,
    group_name: str,
    out_prefix: str,
    dynamic_min_samples: int,
    split_seed: int,
    train_frac: float,
):
    group_safe = sanitize_name(group_name)
    base = f"{out_prefix}/{modality}/{celltype}/{group_level}/{group_safe}"
    local_dir = os.path.join(LOCAL_TMP_ROOT, modality, celltype, group_level, group_safe)
    os.makedirs(local_dir, exist_ok=True)

    array_map = {
        "N": ref["N"].astype(np.float32),
        "keep": ref["keep"].astype(np.uint8),
        "pearson_r": ref["pearson_r"].astype(np.float32),
        "absR": ref["absR"].astype(np.float32),
        "t_stat": ref["t_stat"].astype(np.float32),
        "intercept": ref["intercept"].astype(np.float32),
        "coef": ref["coef"].astype(np.float32),
        "eligible_mask": ref["eligible_mask"].astype(np.uint8),
        "age_grid": age_grid.astype(np.float32),
    }

    for name, arr in array_map.items():
        gs_path = f"{base}/arrays/{name}.npy"
        save_array_local_then_upload(
            arr=arr,
            gs_path=gs_path,
            local_dir=local_dir,
            name=name,
            allow_skip=True
        )
        del arr
        gc.collect()

    eligible_idx = np.where(ref["eligible_mask"])[0].astype(np.int32)

    compact_map = {
        "eligible_idx": eligible_idx,
        "intercept_eligible": ref["intercept"][eligible_idx].astype(np.float32),
        "coef_eligible": ref["coef"][eligible_idx].astype(np.float32),
        "t_stat_eligible": ref["t_stat"][eligible_idx].astype(np.float32),
        "age_grid": age_grid.astype(np.float32),
    }

    for name, arr in compact_map.items():
        gs_path = f"{base}/compact/{name}.npy"
        save_array_local_then_upload(
            arr=arr,
            gs_path=gs_path,
            local_dir=local_dir,
            name=f"compact_{name}",
            allow_skip=True
        )
        del arr
        gc.collect()

    scalars = {
        "EPS": float(ref["EPS"]),
        "CAP": float(ref["CAP"]),
        "ABSR_MIN": float(ABSR_MIN),
        "RANGE_MIN": float(RANGE_MIN),
        "MIN_SAMPLES_PER_SITE": int(MIN_SAMPLES_PER_SITE),
        "DYNAMIC_MIN_FRAC": float(DYNAMIC_MIN_FRAC),
        "dynamic_min_samples": int(dynamic_min_samples),
        "split_seed": int(split_seed),
        "train_frac": float(train_frac),
        "modality": str(modality),
        "celltype": str(celltype),
    }
    upload_text_to_gcs(
        f"{base}/compact/scalars.json",
        json.dumps(scalars, indent=2),
        content_type="application/json",
        timeout=300
    )

    meta_row = {
        "modality": str(modality),
        "celltype": str(celltype),
        "group_level": group_level,
        "group_name": group_name,
        "split_seed": int(split_seed),
        "train_frac": float(train_frac),
        "n_total_samples": int(len(train_df) + len(test_df)),
        "n_train_samples": int(len(train_df)),
        "n_test_samples": int(len(test_df)),
        "age_min_train": float(train_df["Age"].min()),
        "age_max_train": float(train_df["Age"].max()),
        "age_grid_min": float(age_grid.min()),
        "age_grid_max": float(age_grid.max()),
        "age_step": float(AGE_STEP),
        "dynamic_min_samples": int(dynamic_min_samples),
        "n_sites_total": int(len(ref["N"])),
        "n_sites_keep": int(ref["keep"].sum()),
        "n_sites_eligible": int(ref["eligible_mask"].sum()),
        "arrays_prefix": f"{base}/arrays/",
        "compact_prefix": f"{base}/compact/",
        "meta_gs": f"{base}/reference_metadata.csv",
        "train_manifest_gs": f"{base}/train_manifest.csv",
        "test_manifest_gs": f"{base}/test_manifest.csv",
    }

    upload_df_to_gcs_csv(pd.DataFrame([meta_row]), meta_row["meta_gs"])
    upload_df_to_gcs_csv(train_df.copy(), meta_row["train_manifest_gs"])
    upload_df_to_gcs_csv(test_df.copy(), meta_row["test_manifest_gs"])

    upload_text_to_gcs(
        f"{base}/reference_metadata.json",
        json.dumps(meta_row, indent=2),
        content_type="application/json",
        timeout=300
    )

    upsert_reference_index_row(meta_row)
    return meta_row

### Build, split, save, and clear one group

In [ ]:
# ============================================================
# 7) BUILD, SPLIT, SAVE ONE GROUP
# ============================================================
def build_split_save_and_clear_one_group(
    manifest_df: pd.DataFrame,
    modality: str,
    celltype: str,
    group_level: str,
    group_name: str,
    beta_col: str,
    min_group_n: int = 20,
    min_samples_per_site: int = 20,
    dynamic_min_frac: float = 0.50,
    absr_min: float = 0.30,
    range_min: float = 0.05,
    age_step: float = 0.1,
    out_prefix: str = REF_OUT_PREFIX,
    train_frac: float = 0.80,
    split_seed: int = 20260414,
):
    manifest_df = manifest_df.copy().reset_index(drop=True)
    n_group = manifest_df["person_id"].nunique()

    print("\n" + "=" * 100)
    print(f"BUILDING SPLIT REFERENCE | modality={modality} | celltype={celltype} | level={group_level} | group={group_name}")
    print("n unique samples:", n_group)

    if reference_already_exists(modality, celltype, group_level, group_name):
        print(f"Skipping {modality}/{celltype}/{group_level}/{group_name}: already exists in index")
        clear_large_objects(manifest_df)
        return True

    if n_group < min_group_n:
        print(f"Skipping {group_name}: too few samples (< {min_group_n})")
        clear_large_objects(manifest_df)
        return None

    train_df, test_df = deterministic_train_test_split(
        manifest_df,
        train_frac=train_frac,
        seed=split_seed
    )

    n_train_unique = train_df["person_id"].nunique()
    n_test_unique = test_df["person_id"].nunique()

    print(f"Split seed: {split_seed}")
    print(f"Train frac: {train_frac}")
    print(f"Train unique samples: {n_train_unique}")
    print(f"Test  unique samples: {n_test_unique}")

    if n_train_unique < min_group_n:
        print(f"Skipping {group_name}: train subset too small after split (< {min_group_n})")
        clear_large_objects(manifest_df, train_df, test_df)
        return None

    dynamic_min_samples = max(min_samples_per_site, int(dynamic_min_frac * n_train_unique))
    age_grid = build_age_grid_from_train(train_df, age_step=age_step)

    print("Train age range:", float(train_df["Age"].min()), "to", float(train_df["Age"].max()))
    print("Age grid:", float(age_grid.min()), "to", float(age_grid.max()), "n =", len(age_grid))
    print("Dynamic min samples per site:", dynamic_min_samples)

    ref = construct_reference_from_betas(
        train_df,
        beta_col=beta_col,
        min_samples_per_site=dynamic_min_samples
    )

    ref["eligible_mask"] = build_eligible_mask_endpoint_safe(
        ref,
        age_lo=float(age_grid[0]),
        age_hi=float(age_grid[-1]),
        absR_min=absr_min,
        range_min=range_min
    )

    print("Sites keep:", int(ref["keep"].sum()))
    print("Sites eligible:", int(ref["eligible_mask"].sum()))

    meta_row = save_reference_bundle(
        ref=ref,
        train_df=train_df,
        test_df=test_df,
        age_grid=age_grid,
        modality=modality,
        celltype=celltype,
        group_level=group_level,
        group_name=group_name,
        out_prefix=out_prefix,
        dynamic_min_samples=dynamic_min_samples,
        split_seed=split_seed,
        train_frac=train_frac,
    )

    print("Saved:")
    print("  arrays_prefix :", meta_row["arrays_prefix"])
    print("  compact_prefix:", meta_row["compact_prefix"])
    print("  meta          :", meta_row["meta_gs"])
    print("  train         :", meta_row["train_manifest_gs"])
    print("  test          :", meta_row["test_manifest_gs"])
    print("  index         :", REFERENCE_INDEX_GS)

    clear_large_objects(ref, age_grid, meta_row, manifest_df, train_df, test_df)
    return True

### Main execution: build all bulk and cell-type references

In [ ]:
# In TEST mode, manifests are limited to TEST_SAMPLE_LIMIT samples
# In PRODUCTION mode, all samples are used

# ============================================================
# 8) BUILD ALL REFERENCES
# ============================================================
bulk_manifest = discover_bulk_manifest()

print("\nTechnology counts in bulk source manifest:")
display(bulk_manifest.groupby("technology_group")["person_id"].nunique().rename("n_samples").to_frame())

print("\nBatch counts in bulk source manifest:")
display(bulk_manifest.groupby("batch_group")["person_id"].nunique().rename("n_samples").to_frame())

# ---- bulk references ----
build_split_save_and_clear_one_group(
    manifest_df=bulk_manifest[[
        "person_id", "Age", "bulk_beta_path", "source_prefix", "batch_group", "technology_group"
    ]].copy(),
    modality="bulk",
    celltype="bulk",
    group_level="all_samples",
    group_name="all_samples",
    beta_col="bulk_beta_path",
    min_group_n=MIN_GROUP_N,
    min_samples_per_site=MIN_SAMPLES_PER_SITE,
    dynamic_min_frac=DYNAMIC_MIN_FRAC,
    absr_min=ABSR_MIN,
    range_min=RANGE_MIN,
    age_step=AGE_STEP,
    out_prefix=REF_OUT_PREFIX,
    train_frac=TRAIN_FRAC,
    split_seed=SPLIT_SEED,
)

gc.collect()

for tech_name in sorted(bulk_manifest["technology_group"].dropna().unique()):
    sub = bulk_manifest.loc[
        bulk_manifest["technology_group"] == tech_name,
        ["person_id", "Age", "bulk_beta_path", "source_prefix", "batch_group", "technology_group"]
    ].copy()

    build_split_save_and_clear_one_group(
        manifest_df=sub,
        modality="bulk",
        celltype="bulk",
        group_level="technology",
        group_name=tech_name,
        beta_col="bulk_beta_path",
        min_group_n=MIN_GROUP_N,
        min_samples_per_site=MIN_SAMPLES_PER_SITE,
        dynamic_min_frac=DYNAMIC_MIN_FRAC,
        absr_min=ABSR_MIN,
        range_min=RANGE_MIN,
        age_step=AGE_STEP,
        out_prefix=REF_OUT_PREFIX,
        train_frac=TRAIN_FRAC,
        split_seed=SPLIT_SEED,
    )
    clear_large_objects(sub)

for batch_name in sorted(bulk_manifest["batch_group"].dropna().unique()):
    sub = bulk_manifest.loc[
        bulk_manifest["batch_group"] == batch_name,
        ["person_id", "Age", "bulk_beta_path", "source_prefix", "batch_group", "technology_group"]
    ].copy()

    build_split_save_and_clear_one_group(
        manifest_df=sub,
        modality="bulk",
        celltype="bulk",
        group_level="batch",
        group_name=batch_name,
        beta_col="bulk_beta_path",
        min_group_n=MIN_GROUP_N,
        min_samples_per_site=MIN_SAMPLES_PER_SITE,
        dynamic_min_frac=DYNAMIC_MIN_FRAC,
        absr_min=ABSR_MIN,
        range_min=RANGE_MIN,
        age_step=AGE_STEP,
        out_prefix=REF_OUT_PREFIX,
        train_frac=TRAIN_FRAC,
        split_seed=SPLIT_SEED,
    )
    clear_large_objects(sub)

# ---- celltype references ----
for celltype in CELL_TYPES:
    print("\n" + "#" * 120)
    print(f"START CELL TYPE: {celltype}")
    print("#" * 120)

    cell_manifest = build_celltype_manifest_from_bulk(bulk_manifest, celltype)

    if len(cell_manifest) == 0:
        print(f"No files found for cell type {celltype}, skipping.")
        clear_large_objects(cell_manifest)
        continue

    print("\nTechnology counts:")
    display(cell_manifest.groupby("technology_group")["person_id"].nunique().rename("n_samples").to_frame())

    print("\nBatch counts:")
    display(cell_manifest.groupby("batch_group")["person_id"].nunique().rename("n_samples").to_frame())

    build_split_save_and_clear_one_group(
        manifest_df=cell_manifest[[
            "person_id", "Age", "celltype", "celltype_beta_path",
            "source_prefix", "batch_group", "technology_group"
        ]].copy(),
        modality="celltype",
        celltype=celltype,
        group_level="all_samples",
        group_name="all_samples",
        beta_col="celltype_beta_path",
        min_group_n=MIN_GROUP_N,
        min_samples_per_site=MIN_SAMPLES_PER_SITE,
        dynamic_min_frac=DYNAMIC_MIN_FRAC,
        absr_min=ABSR_MIN,
        range_min=RANGE_MIN,
        age_step=AGE_STEP,
        out_prefix=REF_OUT_PREFIX,
        train_frac=TRAIN_FRAC,
        split_seed=SPLIT_SEED,
    )

    gc.collect()

    for tech_name in sorted(cell_manifest["technology_group"].dropna().unique()):
        sub = cell_manifest.loc[
            cell_manifest["technology_group"] == tech_name,
            ["person_id", "Age", "celltype", "celltype_beta_path",
             "source_prefix", "batch_group", "technology_group"]
        ].copy()

        build_split_save_and_clear_one_group(
            manifest_df=sub,
            modality="celltype",
            celltype=celltype,
            group_level="technology",
            group_name=tech_name,
            beta_col="celltype_beta_path",
            min_group_n=MIN_GROUP_N,
            min_samples_per_site=MIN_SAMPLES_PER_SITE,
            dynamic_min_frac=DYNAMIC_MIN_FRAC,
            absr_min=ABSR_MIN,
            range_min=RANGE_MIN,
            age_step=AGE_STEP,
            out_prefix=REF_OUT_PREFIX,
            train_frac=TRAIN_FRAC,
            split_seed=SPLIT_SEED,
        )
        clear_large_objects(sub)

    for batch_name in sorted(cell_manifest["batch_group"].dropna().unique()):
        sub = cell_manifest.loc[
            cell_manifest["batch_group"] == batch_name,
            ["person_id", "Age", "celltype", "celltype_beta_path",
             "source_prefix", "batch_group", "technology_group"]
        ].copy()

        build_split_save_and_clear_one_group(
            manifest_df=sub,
            modality="celltype",
            celltype=celltype,
            group_level="batch",
            group_name=batch_name,
            beta_col="celltype_beta_path",
            min_group_n=MIN_GROUP_N,
            min_samples_per_site=MIN_SAMPLES_PER_SITE,
            dynamic_min_frac=DYNAMIC_MIN_FRAC,
            absr_min=ABSR_MIN,
            range_min=RANGE_MIN,
            age_step=AGE_STEP,
            out_prefix=REF_OUT_PREFIX,
            train_frac=TRAIN_FRAC,
            split_seed=SPLIT_SEED,
        )
        clear_large_objects(sub)

    clear_large_objects(cell_manifest)
    gc.collect()

print("\nFinal reference index:")
final_idx = load_reference_index()
display(final_idx.head(20))
print("Total saved references:", len(final_idx))

### QC: inspect saved reference index

In [ ]:
try:
    final_idx = load_reference_index()
    display(final_idx)
    print(f'Total saved references: {len(final_idx)}')
except Exception as e:
    print('Could not load reference index:', e)
